# Load library

In [29]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="haqESutnQgxgcFePhsfx")
project = rf.workspace("agrlink-d9rrl").project("agrlink-wa29w")
dataset = project.version(2).download("yolov8")



loading Roboflow workspace...
loading Roboflow project...
Dependency ultralytics==8.0.196 is required but found version=8.0.230, to fix: `pip install ultralytics==8.0.196`



Extracting Dataset Version Zip to agrlink-2 in yolov8:: 100%|██████████| 1618/1618 [00:02<00:00, 581.09it/s] 


In [1]:


from roboflow import Roboflow
rf = Roboflow(api_key="HbmRV0tMsZXdeuXIlJWw")
project = rf.workspace("cv-towbw").project("corn-2wdew")
dataset = project.version(17).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...
Dependency ultralytics==8.0.196 is required but found version=8.0.230, to fix: `pip install ultralytics==8.0.196`



Extracting Dataset Version Zip to Corn-17 in yolov8:: 100%|██████████| 1212/1212 [00:00<00:00, 1781.70it/s]


In [ ]:
!pip install albumentations

In [1]:
import albumentations

In [2]:
import wandb
wandb.init(project="AgrlinkTomatoesYOLOe")

wandb: Currently logged in as: pikurovd. Use `wandb login --relogin` to force relogin


In [3]:
import torch

In [4]:
torch.cuda.is_available()

True

In [5]:
import ultralytics

In [21]:
class Albumentations:
    """
    Albumentations transformations.

    Optional, uninstall package to disable. Applies Blur, Median Blur, convert to grayscale, Contrast Limited Adaptive
    Histogram Equalization, random change of brightness and contrast, RandomGamma and lowering of image quality by
    compression.
    """

    def __init__(self, p=1.0):
        """Initialize the transform object for YOLO bbox formatted params."""
        self.p = p
        self.transform = None
        prefix = colorstr('albumentations: ')
        try:
            import albumentations as A
            T = [
                A.RandomShadow(
                  shadow_roi=(0, 0.5, 1, 1),
                  num_shadows_lower=1,
                  num_shadows_upper=2,
                  shadow_dimension=5,
                  always_apply=False,
                  p=0.5),
                A.RandomBrightnessContrast(
                  brightness_limit=0.2,
                  contrast_limit=0.2,
                  brightness_by_max=True,
                  always_apply=False,
                  p=0.5),
                A.GaussNoise(var_limit=(10.0, 50.0), mean=0, per_channel=True, always_apply=False, p=0.5),
                A.ImageCompression(quality_lower=75, p=0.0)]  # transforms
            self.transform = A.Compose(T, bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

            LOGGER.info(prefix + ', '.join(f'{x}'.replace('always_apply=False, ', '') for x in T if x.p))
        except ImportError:  # package not installed, skip
            pass
        except Exception as e:
            LOGGER.info(f'{prefix}{e}')

    def __call__(self, labels):
        """Generates object detections and returns a dictionary with detection results."""
        im = labels['img']
        cls = labels['cls']
        if len(cls):
            labels['instances'].convert_bbox('xywh')
            labels['instances'].normalize(*im.shape[:2][::-1])
            bboxes = labels['instances'].bboxes
            # TODO: add supports of segments and keypoints
            if self.transform and random.random() < self.p:
                new = self.transform(image=im, bboxes=bboxes, class_labels=cls)  # transformed
                if len(new['class_labels']) > 0:  # skip update if no bbox in new im
                    labels['img'] = new['image']
                    labels['cls'] = np.array(new['class_labels'])
                    bboxes = np.array(new['bboxes'], dtype=np.float32)
            labels['instances'].update(bboxes=bboxes)
        return labels



In [6]:
from ultralytics import YOLO
from ultralytics import RTDETR

In [7]:
def tomatoes_yolo():
    model = YOLO("yolov8m-seg.pt", "gpu")
    model.train(
        # Project
        project="AgrlinkTomatoesYOLOe",
        name="yolov8m_seg_v1",
        task='segment',
        # Random Seed parameters
        deterministic=True,
        seed=42,

        # Data & model parameters
        data='C:/Users/Admin/agrlink-2/data.yaml',
        save=True,
        save_period=5,
        pretrained=True,
        imgsz=640,
        # Training parameters
        epochs=80,
        batch=16,
        val=True,
        device='0',
        augment=True,
        workers = 4,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=0.6,
        scale= 0.3,  # image scale (+/- gain)
        fliplr =  0.5,  # image flip left-right (probability)
        mosaic =  0.2,  # image mosaic (probability)
        mixup =  0.2,  # image mixup (probability)
        copy_paste = 0.2,  # segment copy-paste (probability)
        # Optimization parameters
        lr0=0.001,
        patience=10,
        optimizer="AdamW",
        momentum=0.9,
        weight_decay=0.0005,
        close_mosaic=1,
    )

In [8]:
def tomatoes_rtdetr():
    model = RTDETR("rtdetr-x.pt")
    model.train(
        # Project
        project="AgrlinkTomatoesRTDETRe",
        name="rtdetr_xv1",
        task='segment',
        # Random Seed parameters
        deterministic=True,
        seed=42,

        # Data & model parameters
        data='C:/Users/Admin/agrlink-2/data.yaml',
        save=True,
        save_period=5,
        pretrained=True,
        imgsz=640,
        # Training parameters
        epochs=80,
        batch=16,
        val=True,
        device='0',
        augment=True,
        workers = 4,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=0.6,
        scale= 0.3,  # image scale (+/- gain)
        fliplr =  0.5,  # image flip left-right (probability)
        mosaic =  0.2,  # image mosaic (probability)
        mixup =  0.2,  # image mixup (probability)
        copy_paste = 0.2,  # segment copy-paste (probability)
        # Optimization parameters
        lr0=0.001,
        patience=10,
        optimizer="AdamW",
        momentum=0.9,
        weight_decay=0.0005,
        close_mosaic=1,
    )

In [9]:
tomatoes_rtdetr()

100%|██████████| 129M/129M [08:43<00:00, 259kB/s]  


New https://pypi.org/project/ultralytics/8.0.231 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.230 🚀 Python-3.11.5 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
engine\trainer: task=segment, mode=train, model=rtdetr-x.pt, data=C:/Users/Admin/agrlink-2/data.yaml, epochs=80, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=5, cache=False, device=0, workers=4, project=AgrlinkTomatoesRTDETRe, name=rtdetr_xv1, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=1, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=

train: Scanning C:\Users\Admin\agrlink-2\train\labels.cache... 562 images, 0 backgrounds, 0 corrupt: 100%|██████████| 562/562 [00:00<?, ?it/s]

albumentations: RandomShadow(p=0.5, shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1, num_shadows_upper=2, shadow_dimension=5), GaussNoise(p=0.5, var_limit=(10.0, 50.0), per_channel=True, mean=0)



val: Scanning C:\Users\Admin\agrlink-2\valid\labels.cache... 161 images, 0 backgrounds, 0 corrupt: 100%|██████████| 161/161 [00:00<?, ?it/s]


...

  0%|          | 0/36 [00:00<?, ?it/s]C:\Users\Admin\anaconda3\Lib\site-packages\torch\autograd\__init__.py:251: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ..\aten\src\ATen\Context.cpp:75.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
      55/80      20.1G      0.269     0.4617      0.077         15        640: 100%|██████████| 36/36 [01:00<00:00,  1.68s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:04<00:00,  1.39it/s]

                   all        161       2002      0.792      0.756      0.793      0.686



      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/36 [00:00<?, ?it/s]C:\Users\Admin\anaconda3\Lib\site-packages\torch\autograd\__init__.py:251: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ..\aten\src\ATen\Context.cpp:75.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
      56/80      19.3G     0.2479     0.4755    0.06967        220        640:  69%|██████▉   | 25/36 [00:30<00:13,  1.22s/it]


KeyboardInterrupt: 

In [ ]:
tomatoes_yolo()

In [27]:
def cabbage_yolo():
    model = YOLO("yolov8n.pt", "gpu")
    model.train(
        # Project
        project="AgrlinkCabbageYOLOe",
        name="yolov8nv1",
        task='detect',
        # Random Seed parameters
        deterministic=True,
        seed=42,

        # Data & model parameters
        data='C:/Users/Admin/Corn-17/data.yaml',
        save=True,
        save_period=5,
        pretrained=True,
        imgsz=416,
        # Training parameters
        epochs=80,
        batch=16,
        val=True,
        device='0',
        augment=True,
        workers = 4,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=0.6,
        scale= 0.3,  # image scale (+/- gain)
        fliplr =  0.5,  # image flip left-right (probability)
        mosaic =  0.2,  # image mosaic (probability)
        mixup =  0.2,  # image mixup (probability)
        copy_paste = 0.2,  # segment copy-paste (probability)
        # Optimization parameters
        lr0=0.001,
        patience=10,
        optimizer="AdamW",
        momentum=0.9,
        weight_decay=0.0005,
        close_mosaic=1,
    )

In [20]:
def cabbage_rtdetr():
    model = RTDETR("rtdetr-l.pt")
    model.train(
        # Project
        project="AgrlinkCabbageYOLOe",
        name="rtdetr_l",
        task='detect',
        # Random Seed parameters
        deterministic=True,
        seed=42,

        # Data & model parameters
        data='C:/Users/Admin/Corn-17/data.yaml',
        save=True,
        save_period=5,
        pretrained=True,
        imgsz=416,
        # Training parameters
        epochs=80,
        batch=16,
        val=True,
        device='0',
        augment=True,
        workers = 4,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=0.6,
        scale= 0.3,  # image scale (+/- gain)
        fliplr =  0.5,  # image flip left-right (probability)
        mosaic =  0.2,  # image mosaic (probability)
        mixup =  0.2,  # image mixup (probability)
        copy_paste = 0.2,  # segment copy-paste (probability)
        # Optimization parameters
        lr0=0.001,
        patience=10,
        optimizer="AdamW",
        momentum=0.9,
        weight_decay=0.0005,
        close_mosaic=1,
    )

In [ ]:
new_image = '/content/drive/MyDrive/ColabNotebooks/data/3D-EM-Platelet/test/3D-EM-platelet-train04.png'
new_results = my_new_model.predict(new_image, conf=0.2)  #Adjust conf threshold


The results are stored in a variable 'new_results'. Since we only have one image for segmentation, we will only have one set of results. Therefore, let us work with that one result.

In [ ]:
new_result_array = new_results[0].plot()
plt.figure(figsize=(12, 12))
plt.imshow(new_result_array)

**Extracting bounding boxes and segmented masks from the result**

In [ ]:
new_result = new_results[0]

In [ ]:
new_result

**Extracting bounding polygons** <p>
Use 'Masks.xyn' for segments (normalized) and 'Masks.xy' for segments (pixels)

In [ ]:
new_result.masks.xyn

**Extracting segmented masks**

In [ ]:
extracted_masks = new_result.masks.data

In [ ]:
extracted_masks.shape

Push the mask to cpu (from GPU) and convert to numpy array for easy plotting.

In [ ]:
masks_array = extracted_masks.cpu().numpy()

In [ ]:
plt.imshow(masks_array[9])

**Extracting labels for each class**

In [ ]:
class_names = new_result.names.values()
class_names

In [ ]:
# Extract the boxes, which likely contain class IDs
detected_boxes = new_result.boxes.data
# Extract class IDs from the detected boxes
class_labels = detected_boxes[:, -1].int().tolist()
# Initialize a dictionary to hold masks by class
masks_by_class = {name: [] for name in new_result.names.values()}

# Iterate through the masks and class labels
for mask, class_id in zip(extracted_masks, class_labels):
    class_name = new_result.names[class_id]  # Map class ID to class name
    masks_by_class[class_name].append(mask.cpu().numpy())

In [ ]:
for class_name, masks in masks_by_class.items():
    print(f"Class Name: {class_name}, Number of Masks: {len(masks)}")


**Extracting masks for a specific class**

In [ ]:
alpha_granule_masks = masks_by_class['Alpha']
cell_masks = masks_by_class['Cells']

In [ ]:
# Extract the original image
orig_img = new_result.orig_img

In [ ]:
orig_img.shape

In [ ]:
# Display the original image
plt.imshow(orig_img, cmap='gray')

# Overlay the mask with some transparency
#plt.imshow(alpha_granule_masks[1], cmap='jet', alpha=0.3)
plt.imshow(cell_masks[4], cmap='jet', alpha=0.3)
plt.axis('off') # Turn off axis labels
plt.show()

**Calculating region properties for all objects and saving to a csv file.**

In [ ]:
import pandas as pd
from skimage.measure import regionprops

# Initialize a list to store the properties
props_list = []

# Iterate through all classes
for class_name, masks in masks_by_class.items():
    # Iterate through the masks for this class
    for mask in masks:
        # Convert the mask to an integer type if it's not already
        mask = mask.astype(int)

        # Apply regionprops to the mask
        props = regionprops(mask)

        # Extract the properties you want (e.g., area, perimeter) and add them to the list
        for prop in props:
            area = prop.area
            perimeter = prop.perimeter
            # Add other properties as needed

            # Append the properties and class name to the list
            props_list.append({'Class Name': class_name, 'Area': area, 'Perimeter': perimeter})

# Convert the list of dictionaries to a DataFrame
props_df = pd.DataFrame(props_list)

# Now props_df contains the properties and class names for all regions

# Save the DataFrame to a CSV file
props_df.to_csv('/content/drive/MyDrive/ColabNotebooks/data/3D-EM-Platelet/YOLOv8_object_properties.csv', index=False)

In [ ]:
props_df

**Plotting results**

In [ ]:
import seaborn as sns


**Swarm plot**

In [ ]:

# Create the swarm plot with Seaborn
sns.swarmplot(x='Class Name', y='Area', data=props_df)

# Add labels and a title
plt.xlabel('Class Name')
plt.ylabel('Area')
plt.title('Area of Objects for Each Class')

# Rotate the x-axis labels for better visibility if needed
plt.xticks(rotation=45)

# Show the plot
plt.show()

**Box Plot**

In [ ]:
sns.boxplot(x='Class Name', y='Area', data=props_df)
# Add labels and a title
plt.xlabel('Class Name')
plt.ylabel('Area')
plt.title('Area of Objects for Each Class')

# Rotate the x-axis labels for better visibility if needed
plt.xticks(rotation=45)

# Show the plot
plt.show()

**Export model to ONNX for deployment.**

In [ ]:
# Export the model
my_new_model.export(format='onnx', imgsz=[800,800])
